# 离线RAG

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

C:\Users\yanchen\AppData\Local\Temp\ipykernel_31600\3182895319.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


### 读取外部知识

In [6]:
loader = TextLoader(
    "./普通高等学校学生管理规定.txt",
    encoding="utf-8"
)

documents = loader.load()

### 进行分块Chunks

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # 每个文本块的大小
    chunk_overlap=50,  # 文本块之间的重叠部分
)
doc_splits = text_splitter.split_documents(documents)

### 将 chunks 进行 BM25 编码

In [14]:
import jieba
from rank_bm25 import BM25Okapi

pdf_content_words = [jieba.lcut(doc.page_content) for doc in doc_splits]#jieba是用来分词的
bm25_model = BM25Okapi(pdf_content_words)

Building prefix dict from the default dictionary ...
Building prefix dict from the default dictionary ...
Dumping model to file cache C:\Users\yanchen\AppData\Local\Temp\jieba.cache
Dumping model to file cache C:\Users\yanchen\AppData\Local\Temp\jieba.cache
Loading model cost 0.675 seconds.
Loading model cost 0.675 seconds.
Prefix dict has been built successfully.
Prefix dict has been built successfully.


### BM25 编码结果储存

In [16]:
import os
import pickle

os.makedirs("./rag_cache", exist_ok=True)

with open("./rag_cache/bm25_model.pkl", "wb") as f:
    pickle.dump(bm25_model, f)

### 将 chunks 进行向量化

In [4]:
# embedding_model_path = '/data/LLM/maple77/xiaobu-embedding-v2'
embedding_model_path = "BAAI/bge-small-zh-v1.5"
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_path, 
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True} 
)

### 向量化结果存入数据库

In [8]:
vectorstore = Chroma.from_documents(
    documents=doc_splits,
    embedding=embeddings,  
    persist_directory="./rag_cache/chroma_db"
)

In [9]:
from huggingface_hub import snapshot_download

path = snapshot_download(
    repo_id="BAAI/bge-small-zh-v1.5",
    local_files_only=True
)

print(path)

C:\Users\yanchen\.cache\huggingface\hub\models--BAAI--bge-small-zh-v1.5\snapshots\7999e1d3359715c523056ef9478215996d62a620


# 笔记
RAG的笔记详见语雀

通过这部分代码发现自己有很多不懂的：
- 关于模型操作部分，很多参数、环境不懂